# 01 Data Preparation 
### What is done here: 
 - Raw Data from DBPedia is downloaded
 - Raw Data is preprocessed for ingestion in ElasticSearch
 - Connection is generated between DBPedia Id and Wikidata Id
### Data Source: https://downloads.dbpedia.org/2015-10/core/

In [4]:
import re
import requests
import bz2
import pandas as pd


In [5]:
def download_file(url, output_path):
    """Download a file from the specified URL to the output path."""
    response = requests.get(url, stream=True, verify=False)
    if response.status_code == 200:
        with open(output_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=1024):
                file.write(chunk)
        print(f"File downloaded: {output_path}")
    else:
        raise Exception(f"Failed to download file. HTTP Status Code: {response.status_code}")

def extract_bz2(file_path, output_path):
    """Extract a .bz2 file to the specified output path."""
    with bz2.BZ2File(file_path, 'rb') as bz2_file:
        with open(output_path, 'wb') as extracted_file:
            extracted_file.write(bz2_file.read())
        print(f"File extracted: {output_path}")

### Download and prepare the labels

In [15]:
url = "https://downloads.dbpedia.org/2015-10/core/labels_en.ttl.bz2" 
downloaded_file = "../03_data/raw_data/dbpedia/labels_en.ttl.bz2"
extracted_file = "../03_data/raw_data/dbpedia/labels_en.ttl"
download_file(url, downloaded_file)
extract_bz2(downloaded_file, extracted_file)


c:\Users\marvi\.conda\envs\masterthesis\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'downloads.dbpedia.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File downloaded: ../03_data/raw_data/labels_en.ttl.bz2
File extracted: ../03_data/raw_data/labels_en.ttl


In [ ]:
# Read labels File and attach it to a dictionary
labels = []
with open(extracted_file) as file:
    lines = file.readlines()
    for line in lines[1:-1]: # Start and End line contain the timestamp of dbpedia dump.
        labels.append({
            "id": line.split(" ")[0].replace("http://dbpedia.org/resource/","dbpedia:"),
            "title": line.split(">")[2].strip()[1:].replace('"@en .',""),
            "uri": line.split(" ")[0]
        })

In [ ]:

# liste = []
# with open("../raw_data/dbpedia/labels_en.ttl") as file:
#     lines = file.readlines()
#     for line in lines[1:-1]:
#         liste.append({
#             "id": line.split(" ")[0].replace("<http://dbpedia.org/resource/","dbpedia:")[:-1],
#             "title": line.split(">")[2].strip()[1:].replace('"@en .',""),
#             "uri": line.split(" ")[0]
#         })

### Download and prepare the short abstracts

In [21]:
url = "https://downloads.dbpedia.org/2015-10/core/short_abstracts_en.ttl.bz2" 
downloaded_file = "../03_data/raw_data/dbpedia/short_abstracts_en.ttl.bz2"
extracted_file = "../03_data/raw_data/dbpedia/short_abstracts_en.ttl"
download_file(url, downloaded_file)
extract_bz2(downloaded_file, extracted_file)

c:\Users\marvi\.conda\envs\masterthesis\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'downloads.dbpedia.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File downloaded: ../03_data/raw_data/short_abstracts_en.ttl.bz2
File extracted: ../03_data/raw_data/short_abstracts_en.ttl


In [ ]:
abstracts = []
with open(extracted_file) as file:
    lines = file.readlines()
    for line in lines[1:-1]: # Start and End line contain the timestamp of dbpedia dump.
        abstracts.append({
                "id": line.split(" ")[0].replace("http://dbpedia.org/resource/","dbpedia:"),
                "short_abstract": line.split(">")[2].strip()[1:].replace('"@en .',"")
            })

### Merge labels and abstracts to get a preprocessed DataFrame

In [28]:
# Create pandas DataFrames from labels and abstracts
labels = pd.DataFrame(labels)
abstracts = pd.DataFrame(abstracts)
# Remove empty abstract nodes
abstracts = abstracts[abstracts["short_abstract"]!=""]
# Merge labels and abstracts to get a complete DF for the Graph nodes.
# Inner Merge to keep only items that exist in both labels and abstracts
final_df = pd.merge(labels, abstracts,"inner", on="id")
final_df.to_parquet("../03_data/preprocessed_data/nodes_dbpedia.parquet")

### Create Wikidata - DBPedia Relation

In [29]:
url = "https://downloads.dbpedia.org/2015-10/core/page_ids_en.ttl.bz2" 
downloaded_file = "../03_data/raw_data/dbpedia/page_ids_en.ttl.bz2"
extracted_file = "../03_data/raw_data/dbpedia/page_ids_en.ttl"
download_file(url, downloaded_file)
extract_bz2(downloaded_file, extracted_file)

c:\Users\marvi\.conda\envs\masterthesis\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'downloads.dbpedia.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File downloaded: ../03_data/raw_data/page_ids_en.ttl.bz2
File extracted: ../03_data/raw_data/page_ids_en.ttl


In [5]:
mapping = []
with open(extracted_file) as file:
    lines = file.readlines()
    for line in lines[1:-1]:
        mapping.append({
                "id": line.split(" ")[0].replace("http://dbpedia.org/resource/","dbpedia:"),
                "page_id": int(re.search(r'"\d{1,8}"', line).group()[1:-1])
            })

In [6]:
pd.DataFrame(mapping).to_parquet('../03_data/preprocessed_data/mapping_dbpedia_wikidata.parquet')

In [6]:
url = "https://downloads.dbpedia.org/2016-10/core/page_ids_en.ttl.bz2" 
downloaded_file = "../03_data/raw_data/dbpedia/page_ids_en-2016.ttl.bz2"
extracted_file = "../03_data/raw_data/dbpedia/page_ids_en-2016.ttl"
download_file(url, downloaded_file)
extract_bz2(downloaded_file, extracted_file)

c:\Users\marvi\.conda\envs\masterthesis\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'downloads.dbpedia.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


File downloaded: ../03_data/raw_data/dbpedia/page_ids_en-2016.ttl.bz2
File extracted: ../03_data/raw_data/dbpedia/page_ids_en-2016.ttl


In [7]:
mapping = []
with open(extracted_file) as file:
    lines = file.readlines()
    for line in lines[1:-1]:
        mapping.append({
                "id": line.split(" ")[0].replace("http://dbpedia.org/resource/","dbpedia:"),
                "page_id": int(re.search(r'"\d{1,8}"', line).group()[1:-1])
            })

In [8]:
pd.DataFrame(mapping).to_parquet('../03_data/preprocessed_data/mapping_dbpedia_wikidata-2016.parquet')

In [1]:
import pandas as pd

final_df = pd.read_parquet("nodes_dbpedia.parquet")

In [2]:
item_to_index = final_df["uri"].to_dict()
inv_item_to_index = {v: k for k, v in item_to_index.items()}
keys = inv_item_to_index.keys()

In [ ]:
from itertools import islice


In [4]:
rel_df = []
n = 1000000
with open("./page_links_en.ttl/page_links_en.ttl") as file:
    file.readline()
    for n_lines in iter(lambda: tuple(islice(file, n)), ()):
        for i, line in enumerate(n_lines):
            split_line = line.split(" ")
            if split_line[0] in keys and split_line[2] in keys:
                rel_df.append({
                    "left_node": inv_item_to_index[split_line[0]],
                    "right_node": inv_item_to_index[split_line[2]]
            })


In [5]:
len(rel_df)

2

In [5]:
pd.DataFrame(rel_df).to_parquet("rels.parquet")

In [ ]:
rel_df = []
for i, line in enumerate(lines[1:-1]):
    split_line = line.split(" ")
    if split_line[0] in keys and split_line[2] in keys:
        rel_df.append({
            "left_node": inv_item_to_index[split_line[0]],
            "right_node": inv_item_to_index[split_line[2]]
        })
    if i == 100:
        break

In [180]:
split_line = lines[75].split(" ")
len(final_df[final_df["uri"].isin([split_line[0],split_line[2]])])
final_df[final_df["uri"] == split_line[0]].index[0]
right_node = final_df[final_df["uri"] == split_line[2]].index[0]

In [174]:
lines[75]

'<http://dbpedia.org/resource/Albedo> <http://dbpedia.org/ontology/wikiPageWikiLink> <http://dbpedia.org/resource/Johann_Heinrich_Lambert> .\n'

In [184]:
inv_item_to_index

{'<http://dbpedia.org/resource/Albedo>': 0,
 '<http://dbpedia.org/resource/Anarchism>': 1,
 '<http://dbpedia.org/resource/Autism>': 2,
 '<http://dbpedia.org/resource/Achilles>': 3,
 '<http://dbpedia.org/resource/A>': 4,
 '<http://dbpedia.org/resource/Alabama>': 5,
 '<http://dbpedia.org/resource/An_American_in_Paris>': 6,
 '<http://dbpedia.org/resource/Actrius>': 7,
 '<http://dbpedia.org/resource/Animalia_(book)>': 8,
 '<http://dbpedia.org/resource/International_Atomic_Time>': 9,
 '<http://dbpedia.org/resource/Alain_Connes>': 10,
 '<http://dbpedia.org/resource/Allan_Dwan>': 11,
 '<http://dbpedia.org/resource/Academy_Awards>': 12,
 '<http://dbpedia.org/resource/List_of_Atlas_Shrugged_characters>': 13,
 '<http://dbpedia.org/resource/Astronomer>': 14,
 '<http://dbpedia.org/resource/Aristotle>': 15,
 '<http://dbpedia.org/resource/Altruism>': 16,
 '<http://dbpedia.org/resource/Academy_Award_for_Best_Production_Design>': 17,
 '<http://dbpedia.org/resource/ASCII>': 18,
 '<http://dbpedia.org/re

In [169]:
rel_df

[{'left_node': 0, 'right_node': 7819},
 {'left_node': 0, 'right_node': 230908},
 {'left_node': 0, 'right_node': 23905},
 {'left_node': 0, 'right_node': 4628},
 {'left_node': 0, 'right_node': 7937},
 {'left_node': 0, 'right_node': 254811},
 {'left_node': 0, 'right_node': 78332},
 {'left_node': 0, 'right_node': 23598},
 {'left_node': 0, 'right_node': 19785},
 {'left_node': 0, 'right_node': 790400},
 {'left_node': 0, 'right_node': 40199},
 {'left_node': 0, 'right_node': 3834167},
 {'left_node': 0, 'right_node': 30245},
 {'left_node': 0, 'right_node': 21083},
 {'left_node': 0, 'right_node': 3863},
 {'left_node': 0, 'right_node': 883263},
 {'left_node': 0, 'right_node': 2146941},
 {'left_node': 0, 'right_node': 178352},
 {'left_node': 0, 'right_node': 178304},
 {'left_node': 0, 'right_node': 258328},
 {'left_node': 0, 'right_node': 9185},
 {'left_node': 0, 'right_node': 989171},
 {'left_node': 0, 'right_node': 5358},
 {'left_node': 0, 'right_node': 303071},
 {'left_node': 0, 'right_node': 9

In [166]:
final_df.iloc[890507]

id                                         dbpedia:Space_weathering
title                                              Space weathering
uri                  <http://dbpedia.org/resource/Space_weathering>
short_abstract    Space weathering is the damage that occurs to ...
Name: 890507, dtype: object

In [163]:
right_node

890507

In [152]:
len(final_df[final_df["uri"].isin([split_line[0],split_line[2]])])

1

In [156]:
final_df[final_df["uri"]==split_line[0]]


,id,title,uri,short_abstract


In [158]:
abstracts[abstracts["id"]=="dbpedia:AccessibleComputing"]

,id,short_abstract


# Wikilinks

In [20]:
rel_df = []
n = 1000000
with open("./page_ids_en.ttl/page_ids_en.ttl") as file:
    for i in range(1000):
        print(file.readline())
        line = file.readline()

# started 2017-03-14T08:38:43Z

<http://dbpedia.org/resource/AfghanistanHistory> <http://dbpedia.org/ontology/wikiPageID> "13"^^<http://www.w3.org/2001/XMLSchema#integer> .

<http://dbpedia.org/resource/AfghanistanPeople> <http://dbpedia.org/ontology/wikiPageID> "15"^^<http://www.w3.org/2001/XMLSchema#integer> .

<http://dbpedia.org/resource/AfghanistanTransportations> <http://dbpedia.org/ontology/wikiPageID> "19"^^<http://www.w3.org/2001/XMLSchema#integer> .

<http://dbpedia.org/resource/AfghanistanTransnationalIssues> <http://dbpedia.org/ontology/wikiPageID> "21"^^<http://www.w3.org/2001/XMLSchema#integer> .

<http://dbpedia.org/resource/AmoeboidTaxa> <http://dbpedia.org/ontology/wikiPageID> "24"^^<http://www.w3.org/2001/XMLSchema#integer> .

<http://dbpedia.org/resource/AlbaniaHistory> <http://dbpedia.org/ontology/wikiPageID> "27"^^<http://www.w3.org/2001/XMLSchema#integer> .

<http://dbpedia.org/resource/AlbaniaEconomy> <http://dbpedia.org/ontology/wikiPageID> "36"^^<http://www.w3.

In [3]:
import re

In [45]:
int(re.search(r'"\d{1,7}"', line).group()[1:-1])

3509

In [13]:
mapping = []
with open("./page_ids_en.ttl/page_ids_en.ttl") as file:
    lines = file.readlines()
    for line in lines[1:-1]:
        mapping.append({
                "id": line.split(" ")[0],
                "page_id": int(re.search(r'"\d{1,8}"', line).group()[1:-1])
            })

In [10]:
line.split(" ")[0]

'<http://dbpedia.org/resource/Amelia_Ong>'

In [14]:
import pandas as pd
pd.DataFrame(mapping).to_parquet('mapping_dbpedia_wikidata.parquet')